In [1]:
%pip install scikit-learn pandas numpy optuna xgboost lightgbm catboost imbalanced-learn


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# %%
import pandas as pd
import numpy as np
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()

    # --- original features (keep everything from v3) ---
    df['prev_success']        = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted']     = (df['previous'] == 0).astype(int)
    df['pdays_clean']         = df['pdays'].apply(lambda x: 999 if x == -1 else x)
    df['prev_contacts_log']   = np.log1p(df['previous'])
    df['duration_log']        = np.log1p(df['duration'])
    df['log_balance']         = np.log1p(df['balance'].clip(lower=0))
    df['is_debt']             = (df['balance'] < 0).astype(int)
    df['log_campaign']        = np.log1p(df['campaign'])
    df['month_sin']           = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']           = np.cos(2 * np.pi * df['month'] / 12)
    df['long_call']           = (df['duration'] > 300).astype(int)
    df['long_call_x_success'] = df['long_call'] * df['prev_success']
    df['duration_x_prev']     = df['duration_log'] * df['prev_contacts_log']

    # --- NEW interaction features ---
    # duration × prev_success: long call AND previously successful = strong signal
    df['duration_x_success']  = df['duration_log'] * df['prev_success']
    # pdays recency × previous contact count
    df['pdays_x_prev']        = (1 / (df['pdays_clean'] + 1)) * df['prev_contacts_log']
    # balance-to-debt flag interaction
    df['balance_x_debt']      = df['log_balance'] * (1 - df['is_debt'])
    # campaign efficiency: fewer contacts with longer calls = more engaged
    df['duration_per_contact']= df['duration_log'] / (df['log_campaign'] + 1)
    # age binned into young / mid / senior (rough)
    df['age_young']           = (df['age'] < 30).astype(int)
    df['age_senior']          = (df['age'] >= 55).astype(int)

    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)

print('TRAIN_DATA shape:', TRAIN_DATA.shape)
print('TEST_DATA  shape:', TEST_DATA.shape)

TRAIN_DATA shape: (29839, 35)
TEST_DATA  shape: (19893, 35)


/Users/dayana/Desktop/3.1학기/ 기계학습/lab/lab1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# %%
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = [c for c in TRAIN_DATA.columns if c not in cat_cols]

ENCODER = OrdinalEncoder(
    handle_unknown='use_encoded_value', unknown_value=-1
).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]), columns=cat_cols, index=df.index
    )
    return pd.concat([cat_enc, df[num_cols].copy()], axis=1)

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test  = preprocess(TEST_DATA,  ENCODER)
y_train = TRAIN_LABEL['subscription'].values

# --- cross-validated target encoding (same as v3, no leakage) ---
def target_encode_cv(X_tr, y_tr, X_te, cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_tr_enc = X_tr.copy()
    X_te_enc = X_te.copy()
    global_mean = y_tr.mean()
    for col in cols:
        oof    = np.full(len(X_tr), global_mean)
        te_vals = np.zeros(len(X_te))
        for fold_tr_idx, fold_val_idx in skf.split(X_tr, y_tr):
            means = y_tr.iloc[fold_tr_idx].groupby(X_tr[col].iloc[fold_tr_idx]).mean()
            oof[fold_val_idx] = X_tr[col].iloc[fold_val_idx].map(means).fillna(global_mean).values
            te_vals += X_te[col].reset_index(drop=True).map(means).fillna(global_mean).values / n_splits
        X_tr_enc[col + '_te'] = oof
        X_te_enc[col + '_te'] = te_vals
    return X_tr_enc, X_te_enc

y_series = pd.Series(y_train, index=X_train.index)
X_train_te, X_test_te = target_encode_cv(X_train, y_series, X_test, cat_cols)

print('X_train_te shape:', X_train_te.shape)
print('X_test_te  shape:', X_test_te.shape)

X_train_te shape: (29839, 43)
X_test_te  shape: (19893, 43)


In [4]:
# %%
# ================================================================
#  OPTUNA TUNING  — XGBoost, LightGBM (incl. dart), CatBoost
#  HGBM dropped: it received 0 weight in v3 ensemble
# ================================================================
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

N_TRIALS    = 150   # more trials → better params (takes ~2-3× v3 time)
CV_SPLITS   = 5
RANDOM_SEED = 42

scale_pos  = (y_train == 0).sum() / (y_train == 1).sum()
X_arr      = X_train_te.values
X_te_arr   = X_test_te.values
inner_skf  = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)

def youden_ba_cv(model, X, y, cv):
    """OOF BA using Youden-J optimal threshold."""
    oof = np.zeros(len(y))
    for tr_idx, val_idx in cv.split(X, y):
        model.fit(X[tr_idx], y[tr_idx])
        oof[val_idx] = model.predict_proba(X[val_idx])[:, 1]
    fpr, tpr, ths = roc_curve(y, oof)
    best_t = float(ths[np.argmax(tpr - fpr)])
    return balanced_accuracy_score(y, (oof >= best_t).astype(int))

# ----------------------------------------------------------------
#  XGBoost
# ----------------------------------------------------------------
def xgb_objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int(  'n_estimators',    300,  2000),
        'learning_rate'   : trial.suggest_float('learning_rate',   0.005, 0.15, log=True),
        'max_depth'       : trial.suggest_int(  'max_depth',       3,    10),
        'min_child_weight': trial.suggest_int(  'min_child_weight',3,    80),
        'subsample'       : trial.suggest_float('subsample',       0.5,  1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.4,  1.0),
        'colsample_bylevel':trial.suggest_float('colsample_bylevel',0.4, 1.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha',       1e-3, 20.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda',      1e-3, 20.0, log=True),
        'gamma'           : trial.suggest_float('gamma',           0.0,  5.0),
        'scale_pos_weight': scale_pos,
        'eval_metric'     : 'logloss',
        'random_state'    : RANDOM_SEED,
        'n_jobs'          : -1,
    }
    return youden_ba_cv(XGBClassifier(**params), X_arr, y_train, inner_skf)

print('Tuning XGBoost (150 trials)...')
xgb_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_xgb = xgb_study.best_params
print(f'  Best XGB  BA: {xgb_study.best_value:.5f}')
print(f'  Params: {best_xgb}')

# ----------------------------------------------------------------
#  LightGBM  —  gbdt + dart both searched
# ----------------------------------------------------------------
def lgbm_objective(trial):
    booster = trial.suggest_categorical('boosting_type', ['gbdt', 'dart'])
    params = {
        'boosting_type'   : booster,
        'n_estimators'    : trial.suggest_int(  'n_estimators',    200,  2000),
        'learning_rate'   : trial.suggest_float('learning_rate',   0.005, 0.15, log=True),
        'max_depth'       : trial.suggest_int(  'max_depth',       3,    12),
        'num_leaves'      : trial.suggest_int(  'num_leaves',      15,   200),
        'min_child_samples': trial.suggest_int( 'min_child_samples',5,   120),
        'subsample'       : trial.suggest_float('subsample',       0.5,  1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.4,  1.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha',       1e-3, 20.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda',      1e-3, 20.0, log=True),
        'class_weight'    : 'balanced',
        'random_state'    : RANDOM_SEED,
        'n_jobs'          : -1,
        'verbose'         : -1,
    }
    # dart-specific params
    if booster == 'dart':
        params['drop_rate'] = trial.suggest_float('drop_rate', 0.05, 0.3)
        params['skip_drop'] = trial.suggest_float('skip_drop', 0.3,  0.7)
    return youden_ba_cv(LGBMClassifier(**params), X_arr, y_train, inner_skf)

print('\nTuning LightGBM (150 trials, gbdt + dart)...')
lgbm_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_lgbm = lgbm_study.best_params
print(f'  Best LGBM BA: {lgbm_study.best_value:.5f}')
print(f'  Params: {best_lgbm}')

# ----------------------------------------------------------------
#  CatBoost  —  wider depth + border count search
# ----------------------------------------------------------------
def cat_objective(trial):
    params = {
        'iterations'         : trial.suggest_int(  'iterations',         300,  2500),
        'learning_rate'      : trial.suggest_float('learning_rate',      0.005, 0.15, log=True),
        'depth'              : trial.suggest_int(  'depth',              3,    10),
        'l2_leaf_reg'        : trial.suggest_float('l2_leaf_reg',        0.5,  20.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature',0.0,  1.5),
        'random_strength'    : trial.suggest_float('random_strength',    0.0,  8.0),
        'border_count'       : trial.suggest_int(  'border_count',       32,   255),
        'auto_class_weights' : 'Balanced',
        'eval_metric'        : 'Logloss',
        'random_seed'        : RANDOM_SEED,
        'verbose'            : 0,
    }
    return youden_ba_cv(CatBoostClassifier(**params), X_arr, y_train, inner_skf)

print('\nTuning CatBoost (150 trials)...')
cat_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
cat_study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_cat = cat_study.best_params
print(f'  Best CAT  BA: {cat_study.best_value:.5f}')
print(f'  Params: {best_cat}')

print('\n' + '='*50)
print('OPTUNA SUMMARY')
print('='*50)
print(f'  XGB  : {xgb_study.best_value:.5f}')
print(f'  LGBM : {lgbm_study.best_value:.5f}')
print(f'  CAT  : {cat_study.best_value:.5f}')

Tuning XGBoost (150 trials)...


Best trial: 129. Best value: 0.872534: 100%|██████████| 150/150 [40:16<00:00, 16.11s/it]


  Best XGB  BA: 0.87253
  Params: {'n_estimators': 1398, 'learning_rate': 0.014688107933307866, 'max_depth': 9, 'min_child_weight': 11, 'subsample': 0.9081675632335575, 'colsample_bytree': 0.9154397069234146, 'colsample_bylevel': 0.7859644524805499, 'reg_alpha': 14.449527198623775, 'reg_lambda': 0.0013131988451547652, 'gamma': 1.3271607503475447}

Tuning LightGBM (150 trials, gbdt + dart)...


Best trial: 128. Best value: 0.873407: 100%|██████████| 150/150 [5:07:14<00:00, 122.89s/it]   


  Best LGBM BA: 0.87341
  Params: {'boosting_type': 'gbdt', 'n_estimators': 1544, 'learning_rate': 0.009776236382241175, 'max_depth': 12, 'num_leaves': 189, 'min_child_samples': 79, 'subsample': 0.6758166608075022, 'colsample_bytree': 0.6287663520283125, 'reg_alpha': 14.059097249147474, 'reg_lambda': 0.5598467566307593}

Tuning CatBoost (150 trials)...


Best trial: 145. Best value: 0.872562: 100%|██████████| 150/150 [5:10:31<00:00, 124.21s/it]   

  Best CAT  BA: 0.87256
  Params: {'iterations': 639, 'learning_rate': 0.013197977619438363, 'depth': 10, 'l2_leaf_reg': 9.52653108639879, 'bagging_temperature': 1.422724713850696, 'random_strength': 0.14507304848953856, 'border_count': 167}

OPTUNA SUMMARY
  XGB  : 0.87253
  LGBM : 0.87341
  CAT  : 0.87256


In [5]:
# %%
# ================================================================
#  OOF + TEST PREDICTIONS  —  10-fold, 3 models
# ================================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_curve

N_SPLITS   = 10
skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
model_names= ['XGB', 'LGBM', 'CAT']

oof_preds  = {name: np.zeros(len(y_train))    for name in model_names}
test_preds = {name: np.zeros(len(X_test_te))  for name in model_names}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    # XGB
    xgb_p = {**best_xgb,
              'scale_pos_weight': scale_pos,
              'eval_metric'     : 'logloss',
              'random_state'    : RANDOM_SEED,
              'n_jobs'          : -1}
    m = XGBClassifier(**xgb_p)
    m.fit(X_tr, y_tr)
    oof_preds['XGB'][val_idx]   = m.predict_proba(X_val)[:, 1]
    test_preds['XGB']          += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    # LGBM
    lgbm_p = {**best_lgbm,
               'class_weight': 'balanced',
               'random_state': RANDOM_SEED,
               'n_jobs'      : -1,
               'verbose'     : -1}
    m = LGBMClassifier(**lgbm_p)
    m.fit(X_tr, y_tr)
    oof_preds['LGBM'][val_idx]  = m.predict_proba(X_val)[:, 1]
    test_preds['LGBM']         += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    # CatBoost
    cat_p = {**best_cat,
              'auto_class_weights': 'Balanced',
              'eval_metric'       : 'Logloss',
              'random_seed'       : RANDOM_SEED,
              'verbose'           : 0}
    m = CatBoostClassifier(**cat_p)
    m.fit(X_tr, y_tr)
    oof_preds['CAT'][val_idx]   = m.predict_proba(X_val)[:, 1]
    test_preds['CAT']          += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    print(f'Fold {fold+1}/{N_SPLITS} done')

print('\nOOF Balanced Accuracy per model (Youden J threshold):')
for name in model_names:
    fpr, tpr, thresholds = roc_curve(y_train, oof_preds[name])
    best_t  = float(thresholds[np.argmax(tpr - fpr)])
    best_ba = balanced_accuracy_score(y_train, (oof_preds[name] >= best_t).astype(int))
    print(f'  {name:5s}: BA={best_ba:.5f}  threshold={best_t:.4f}')

Fold 1/10 done
Fold 2/10 done
Fold 3/10 done
Fold 4/10 done
Fold 5/10 done
Fold 6/10 done
Fold 7/10 done
Fold 8/10 done
Fold 9/10 done
Fold 10/10 done

OOF Balanced Accuracy per model (Youden J threshold):
  XGB  : BA=0.87620  threshold=0.3644
  LGBM : BA=0.87503  threshold=0.4534
  CAT  : BA=0.87506  threshold=0.3743


In [6]:
# %%
# ================================================================
#  ENSEMBLE  —  BA-weighted average (simpler & more robust than
#  a meta-learner when we only have 3 base models)
# ================================================================
from sklearn.metrics import balanced_accuracy_score, roc_curve
from scipy.stats import rankdata

# Rank-normalise so each model is on the same scale
def rank_norm(arr):
    return rankdata(arr) / len(arr)

# Per-model OOF BA  (used for weighting)
oof_ba_scores = {}
print('Individual OOF BA scores:')
for name in model_names:
    fpr, tpr, ths = roc_curve(y_train, oof_preds[name])
    t  = float(ths[np.argmax(tpr - fpr)])
    ba = balanced_accuracy_score(y_train, (oof_preds[name] >= t).astype(int))
    oof_ba_scores[name] = ba
    print(f'  {name}: {ba:.5f}')

# Softmax-style weighting (amplifies differences between models)
raw_w = np.array([oof_ba_scores[n] for n in model_names])
# Shift and amplify before normalising so better models dominate more
shifted = raw_w - raw_w.min()
weights  = shifted ** 2          # square to amplify gaps
weights  = weights / weights.sum()
print('\nModel weights (BA²-proportional):')
for n, w in zip(model_names, weights):
    print(f'  {n}: {w:.4f}')

# Rank-norm then weighted average
oof_rank  = np.column_stack([rank_norm(oof_preds[n])  for n in model_names])
test_rank = np.column_stack([rank_norm(test_preds[n]) for n in model_names])

oof_blend  = oof_rank  @ weights
test_blend = test_rank @ weights

# Youden J threshold on blended OOF
fpr_b, tpr_b, thresholds_b = roc_curve(y_train, oof_blend)
j_b             = tpr_b - fpr_b
best_threshold  = float(thresholds_b[np.argmax(j_b)])
best_ba         = balanced_accuracy_score(
    y_train, (oof_blend >= best_threshold).astype(int)
)

print(f'\nBlended OOF BA       : {best_ba:.5f}')
print(f'Optimal threshold    : {best_threshold:.4f}')

# Neighbourhood scan so we can pick the best threshold manually if needed
print('\nThreshold | #Pred-1 | OOF BA')
print('-' * 38)
for t in np.arange(
    max(0.01, best_threshold - 0.05),
    min(0.99, best_threshold + 0.06),
    0.005
):
    preds = (oof_blend >= t).astype(int)
    ba    = balanced_accuracy_score(y_train, preds)
    mark  = ' <- best' if abs(t - best_threshold) < 0.003 else ''
    print(f'  {t:.3f}  |  {preds.sum():6d}  |  {ba:.4f}{mark}')

Individual OOF BA scores:
  XGB: 0.87620
  LGBM: 0.87503
  CAT: 0.87506

Model weights (BA²-proportional):
  XGB: 0.9994
  LGBM: 0.0000
  CAT: 0.0006

Blended OOF BA       : 0.87620
Optimal threshold    : 0.7537

Threshold | #Pred-1 | OOF BA
--------------------------------------
  0.704  |    8841  |  0.8661
  0.709  |    8692  |  0.8676
  0.714  |    8542  |  0.8687
  0.719  |    8393  |  0.8697
  0.724  |    8243  |  0.8701
  0.729  |    8095  |  0.8711
  0.734  |    7944  |  0.8725
  0.739  |    7796  |  0.8737
  0.744  |    7647  |  0.8748
  0.749  |    7499  |  0.8750
  0.754  |    7349  |  0.8762 <- best
  0.759  |    7200  |  0.8755
  0.764  |    7051  |  0.8739
  0.769  |    6901  |  0.8738
  0.774  |    6753  |  0.8722
  0.779  |    6603  |  0.8710
  0.784  |    6454  |  0.8708
  0.789  |    6304  |  0.8687
  0.794  |    6155  |  0.8673
  0.799  |    6007  |  0.8666
  0.804  |    5858  |  0.8640
  0.809  |    5708  |  0.8622
  0.814  |    5558  |  0.8577


In [7]:
# %%
# ================================================================
#  GENERATE SUBMISSION
# ================================================================
test_classes = (test_blend >= best_threshold).astype(int)
n1 = test_classes.sum()
n0 = (test_classes == 0).sum()

print(f'Prediction distribution — 0: {n0}, 1: {n1} (pos-rate {n1/(n0+n1)*100:.1f}%)')
print(f'Blended OOF BA  : {best_ba:.5f}')
print(f'Threshold used  : {best_threshold:.4f}')

submission = pd.DataFrame({
    'id'          : TEST_DATA.index,
    'subscription': test_classes
})
submission.to_csv('submission_optuna_v4.csv', index=False)
print('\nSaved submission_optuna_v4.csv  ✓')
print(submission.head())
print(submission['subscription'].value_counts())

Prediction distribution — 0: 14994, 1: 4899 (pos-rate 24.6%)
Blended OOF BA  : 0.87620
Threshold used  : 0.7537

Saved submission_optuna_v4.csv  ✓
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             1
subscription
0    14994
1     4899
Name: count, dtype: int64
